# Batch Processing for AI Pipelines

This notebook covers practical batch processing techniques:
- Batch inference patterns
- Processing multiple documents
- Async processing with asyncio
- When to batch vs stream
- Performance comparison

## 1. Setup

In [ ]:
import asyncio
import time
from dataclasses import dataclass, field
from typing import Any, Callable

print("Dependencies loaded.")

## 2. Sequential vs Parallel Processing

Baseline comparison: sequential processing vs async parallel.

In [ ]:
async def simulate_api_call(item: str, latency: float = 0.1) -> str:
    """Simulate an API call with latency."""
    await asyncio.sleep(latency)
    return f"Processed: {item}"


async def process_sequential(items: list[str]) -> list[str]:
    """Process items one at a time (baseline)."""
    results = []
    for item in items:
        result = await simulate_api_call(item)
        results.append(result)
    return results


async def process_parallel(items: list[str], max_concurrent: int = 10) -> list[str]:
    """Process items in parallel with concurrency limit."""
    semaphore = asyncio.Semaphore(max_concurrent)

    async def process_with_limit(item: str) -> str:
        async with semaphore:
            return await simulate_api_call(item)

    tasks = [process_with_limit(item) for item in items]
    return await asyncio.gather(*tasks)


# Benchmark
items = [f"item_{i}" for i in range(20)]

print("Sequential processing...")
start = time.time()
results_seq = await process_sequential(items)
time_seq = time.time() - start
print(f"  Time: {time_seq:.2f}s for {len(items)} items")

print("\nParallel processing (max 5 concurrent)...")
start = time.time()
results_par = await process_parallel(items, max_concurrent=5)
time_par = time.time() - start
print(f"  Time: {time_par:.2f}s for {len(items)} items")

print(f"\nSpeedup: {time_seq / time_par:.1f}x")

## 3. Batch Inference Patterns

Group multiple inputs into a single API call to reduce overhead.

In [ ]:
@dataclass
class BatchResult:
    items: list[str]
    results: list[str]
    latency_ms: float
    api_calls: int


async def batch_generate(
    prompts: list[str],
    batch_size: int = 10,
    process_fn: Callable = None
) -> BatchResult:
    """Process prompts in batches."""
    if process_fn is None:
        process_fn = simulate_api_call

    all_results = []
    api_calls = 0
    start = time.time()

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        # Process batch concurrently
        batch_results = await asyncio.gather(*[process_fn(p) for p in batch])
        all_results.extend(batch_results)
        api_calls += 1

    latency = (time.time() - start) * 1000
    return BatchResult(
        items=prompts,
        results=all_results,
        latency_ms=latency,
        api_calls=api_calls
    )


# Demo
prompts = [
    "Summarize this document.",
    "Translate to Spanish.",
    "Extract key entities.",
    "Classify sentiment.",
    "Generate a title.",
    "Summarize this document.",
    "Translate to French.",
    "Extract key entities.",
]

result = await batch_generate(prompts, batch_size=4)
print(f"Processed {len(result.items)} items in {result.latency_ms:.0f}ms")
print(f"API calls made: {result.api_calls}")
for r in result.results:
    print(f"  {r}")

## 4. Document Processing Pipeline

Process a batch of documents through multiple stages.

In [ ]:
@dataclass
class Document:
    id: str
    content: str
    metadata: dict = field(default_factory=dict)


@dataclass
class ProcessedDocument:
    id: str
    summary: str
    embedding: list[float]
    entities: list[str]


class DocumentProcessor:
    """Multi-stage document processing pipeline."""

    def __init__(self, max_concurrent: int = 5):
        self.max_concurrent = max_concurrent

    async def _summarize(self, doc: Document) -> str:
        """Simulate LLM summarization."""
        await asyncio.sleep(0.05)
        return f"Summary of {doc.id}: {doc.content[:50]}..."

    async def _embed(self, text: str) -> list[float]:
        """Simulate embedding computation."""
        await asyncio.sleep(0.02)
        import random
        rng = random.Random(hash(text))
        return [rng.random() for _ in range(384)]

    async def _extract_entities(self, doc: Document) -> list[str]:
        """Simulate entity extraction."""
        await asyncio.sleep(0.03)
        return [word for word in doc.content.split() if word[0].isupper()]

    async def process_single(self, doc: Document) -> ProcessedDocument:
        """Process a single document through all stages."""
        # Run stages concurrently where possible
        summary_task = self._summarize(doc)
        embed_task = self._embed(doc.content)
        entity_task = self._extract_entities(doc)

        summary, embedding, entities = await asyncio.gather(
            summary_task, embed_task, entity_task
        )

        return ProcessedDocument(
            id=doc.id,
            summary=summary,
            embedding=embedding,
            entities=entities
        )

    async def process_batch(self, documents: list[Document]) -> list[ProcessedDocument]:
        """Process multiple documents with concurrency control."""
        semaphore = asyncio.Semaphore(self.max_concurrent)

        async def process_with_limit(doc: Document) -> ProcessedDocument:
            async with semaphore:
                return await self.process_single(doc)

        tasks = [process_with_limit(doc) for doc in documents]
        return await asyncio.gather(*tasks)


# Demo
documents = [
    Document(id="doc_1", content="Python is a programming language used for AI."),
    Document(id="doc_2", content="Machine learning requires large datasets for training."),
    Document(id="doc_3", content="Neural networks are inspired by the human brain."),
    Document(id="doc_4", content="Deep learning has revolutionized natural language processing."),
    Document(id="doc_5", content="Transformers are the foundation of modern NLP models."),
]

processor = DocumentProcessor(max_concurrent=3)

print("Processing documents...")
start = time.time()
results = await processor.process_batch(documents)
elapsed = time.time() - start

print(f"\nProcessed {len(results)} documents in {elapsed:.2f}s")
for doc in results:
    print(f"\n{doc.id}:")
    print(f"  Summary: {doc.summary}")
    print(f"  Entities: {doc.entities}")
    print(f"  Embedding dim: {len(doc.embedding)}")

## 5. Async Processing Patterns

Common patterns for building async pipelines.

In [ ]:
# Pattern 1: Producer-Consumer with Queue

async def producer_consumer_pattern():
    queue = asyncio.Queue(maxsize=10)
    results = []

    async def producer():
        for i in range(20):
            await queue.put(f"item_{i}")
        # Signal completion
        for _ in range(3):  # 3 consumers
            await queue.put(None)

    async def consumer(name: str):
        while True:
            item = await queue.get()
            if item is None:
                break
            # Process item
            await asyncio.sleep(0.05)
            results.append(f"{name} processed {item}")
            queue.task_done()

    # Start producer and consumers
    await asyncio.gather(
        producer(),
        consumer("worker_1"),
        consumer("worker_2"),
        consumer("worker_3")
    )

    return results


results = await producer_consumer_pattern()
print(f"Producer-consumer: {len(results)} items processed")

In [ ]:
# Pattern 2: Rate-limited batch processing

class RateLimitedBatcher:
    """Process items with rate limiting."""

    def __init__(self, rate: float = 10.0, burst: int = 20):
        self.rate = rate  # items per second
        self.burst = burst
        self.tokens = burst
        self.last_refill = time.time()

    async def _refill(self):
        now = time.time()
        elapsed = now - self.last_refill
        self.tokens = min(self.burst, self.tokens + elapsed * self.rate)
        self.last_refill = now

    async def process(self, items: list[str], process_fn: Callable) -> list[str]:
        results = []
        for item in items:
            await self._refill()
            if self.tokens < 1:
                wait_time = (1 - self.tokens) / self.rate
                await asyncio.sleep(wait_time)
                await self._refill()
            self.tokens -= 1
            result = await process_fn(item)
            results.append(result)
        return results


batcher = RateLimitedBatcher(rate=10, burst=5)
items = [f"item_{i}" for i in range(15)]

start = time.time()
results = await batcher.process(items, simulate_api_call)
elapsed = time.time() - start

print(f"Rate-limited: {len(results)} items in {elapsed:.2f}s")

In [ ]:
# Pattern 3: Pipeline with stages

async def pipeline_pattern():
    """Multi-stage pipeline with bounded buffers."""
    stage1_to_2 = asyncio.Queue(maxsize=5)
    stage2_to_3 = asyncio.Queue(maxsize=5)
    results = []

    async def stage1():
        """Fetch/prepare data."""
        for i in range(15):
            await stage1_to_2.put(f"raw_{i}")
        await stage1_to_2.put(None)

    async def stage2():
        """Process/transform."""
        while True:
            item = await stage1_to_2.get()
            if item is None:
                await stage2_to_3.put(None)
                break
            await asyncio.sleep(0.02)  # processing
            await stage2_to_3.put(f"processed_{item}")

    async def stage3():
        """Store/output."""
        while True:
            item = await stage2_to_3.get()
            if item is None:
                break
            await asyncio.sleep(0.01)  # output
            results.append(item)

    await asyncio.gather(stage1(), stage2(), stage3())
    return results


results = await pipeline_pattern()
print(f"Pipeline: {len(results)} items processed")
print(f"Sample: {results[:3]}")

## 6. Batch vs Stream Decision Guide

Choose the right processing mode based on requirements.

In [ ]:
# Decision matrix
decision_guide = {
    "batch": {
        "when": [
            "High throughput needed",
            "Latency tolerance > 1s",
            "Cost sensitivity",
            "Background processing",
            "Bulk data operations",
        ],
        "examples": [
            "Document ingestion",
            "Nightly analytics",
            "Model training",
            "Data enrichment",
        ],
        "tools": ["asyncio.gather", "Celery", "BullMQ"],
    },
    "stream": {
        "when": [
            "Low latency required",
            "User-facing responses",
            "Progressive output",
            "Real-time updates",
        ],
        "examples": [
            "Chat interfaces",
            "Live search results",
            "Code generation",
            "Real-time dashboards",
        ],
        "tools": ["async generators", "SSE", "WebSockets"],
    },
    "hybrid": {
        "when": [
            "Mixed workloads",
            "Background + real-time",
            "Cost optimization",
        ],
        "examples": [
            "RAG with cached results",
            "Batch precompute + real-time lookup",
            "Queue + streaming response",
        ],
        "tools": ["Redis queues", "Kafka", "BullMQ"],
    },
}

for mode, details in decision_guide.items():
    print(f"\n{'='*50}")
    print(f"{mode.upper()} PROCESSING")
    print(f"{'='*50}")
    print("When:")
    for w in details["when"]:
        print(f"  - {w}")
    print("Examples:")
    for e in details["examples"]:
        print(f"  - {e}")

## 7. Performance Comparison

Benchmark different processing strategies.

In [ ]:
async def benchmark_strategies(n_items: int = 50):
    """Compare processing strategies."""
    items = [f"item_{i}" for i in range(n_items)]

    # Strategy 1: Sequential
    start = time.time()
    await process_sequential(items)
    t_sequential = time.time() - start

    # Strategy 2: Parallel (no limit)
    start = time.time()
    await asyncio.gather(*[simulate_api_call(i) for i in items])
    t_unlimited = time.time() - start

    # Strategy 3: Parallel (limited)
    start = time.time()
    await process_parallel(items, max_concurrent=10)
    t_limited = time.time() - start

    # Strategy 4: Batched
    start = time.time()
    await batch_generate(items, batch_size=10)
    t_batched = time.time() - start

    print(f"Benchmark: {n_items} items, 0.1s simulated latency each")
    print(f"{'='*50}")
    print(f"Sequential:          {t_sequential:.2f}s")
    print(f"Parallel (unlimited): {t_unlimited:.2f}s")
    print(f"Parallel (limited):  {t_limited:.2f}s")
    print(f"Batched:             {t_batched:.2f}s")
    print(f"{'='*50}")
    print(f"Speedup (limited vs seq): {t_sequential / t_limited:.1f}x")

    return {
        "sequential": t_sequential,
        "unlimited": t_unlimited,
        "limited": t_limited,
        "batched": t_batched,
    }


results = await benchmark_strategies()

## Key Takeaways

1. **Async parallel** processing provides significant speedup over sequential
2. **Concurrency limits** prevent overwhelming APIs and respect rate limits
3. **Multi-stage pipelines** with queues handle backpressure naturally
4. **Batch vs stream** depends on latency requirements and cost sensitivity
5. **Producer-consumer** pattern scales well for heterogeneous workloads